# E. Adam & F. SGD Check

**Adam**: E1-E5 | **SGD**: F1-F3

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import inspect
from network import Network, NetworkConfig
from optimizer import Adam, Sgd

## E1: Adam Default Hyperparameters

In [ ]:
adam = Adam(learning_rate=0.001)
print(f'beta1:   {adam.beta1}   (expected: 0.9)   {"PASS" if adam.beta1 == 0.9 else "FAIL"}')
print(f'beta2:   {adam.beta2}  (expected: 0.999) {"PASS" if adam.beta2 == 0.999 else "FAIL"}')
print(f'epsilon: {adam.epsilon}  (expected: 1e-8)  {"PASS" if adam.epsilon == 1e-8 else "FAIL"}')

## E2: m, v Initialized to Zero

In [ ]:
np.random.seed(42)
config = NetworkConfig(layers=[4, 3, 2], activation='relu', loss='cross_entropy',
    output_activation='softmax', weights_initializer='heUniform')
net = Network(config)
adam = Adam(learning_rate=0.001)

print(f'Before update - m_w: {adam.m_w} {"PASS" if adam.m_w is None else "FAIL"}')

x = np.random.randn(3, 4); y = np.array([[1,0],[0,1],[1,0]])
net.forward(x)
nw, nb = net.backward(y)
adam.update(net, nw, nb)

shapes_ok = all(adam.m_w[i].shape == net.weights[i].shape for i in range(len(net.weights)))
print(f'After update - shapes correct: {shapes_ok} {"PASS" if shapes_ok else "FAIL"}')

## E3: Timestep Increments

In [ ]:
adam = Adam(learning_rate=0.001)
np.random.seed(42)
config = NetworkConfig(layers=[4, 3, 2], activation='relu', loss='cross_entropy',
    output_activation='softmax', weights_initializer='heUniform')
net = Network(config)
x = np.random.randn(3, 4); y = np.array([[1,0],[0,1],[1,0]])

steps = []
for _ in range(5):
    net.forward(x)
    nw, nb = net.backward(y)
    adam.update(net, nw, nb)
    steps.append(adam.timestep)

print(f'Timesteps: {steps}')
print(f'Expected:  [1, 2, 3, 4, 5]')
print(f'Status: {"PASS" if steps == [1, 2, 3, 4, 5] else "FAIL"}')

## E4: Bias Correction

In [ ]:
adam = Adam(learning_rate=0.001)
# At t=1: m_hat = m / (1 - 0.9^1) = m / 0.1 = 10x
# At t=1: v_hat = v / (1 - 0.999^1) = v / 0.001 = 1000x
factor_m = 1 / (1 - 0.9 ** 1)
factor_v = 1 / (1 - 0.999 ** 1)

print(f'm correction at t=1: {factor_m:.1f}x (expected: 10.0x) {"PASS" if abs(factor_m - 10.0) < 0.01 else "FAIL"}')
print(f'v correction at t=1: {factor_v:.1f}x (expected: 1000.0x) {"PASS" if abs(factor_v - 1000.0) < 1 else "FAIL"}')

## E5: Per-Layer Independent m, v

In [ ]:
adam = Adam(learning_rate=0.001)
np.random.seed(42)
config = NetworkConfig(layers=[4, 5, 3, 2], activation='relu', loss='cross_entropy',
    output_activation='softmax', weights_initializer='heUniform')
net = Network(config)
x = np.random.randn(3, 4); y = np.array([[1,0],[0,1],[1,0]])
net.forward(x)
nw, nb = net.backward(y)
adam.update(net, nw, nb)

n_layers = len(net.weights)
print(f'Network layers: {n_layers} | m_w entries: {len(adam.m_w)} | v_w entries: {len(adam.v_w)}')
for i in range(n_layers):
    ok = (net.weights[i].shape == adam.m_w[i].shape == adam.v_w[i].shape and
          net.biases[i].shape == adam.m_b[i].shape == adam.v_b[i].shape)
    print(f'  Layer {i}: w{net.weights[i].shape} m{adam.m_w[i].shape} v{adam.v_w[i].shape} | {"PASS" if ok else "FAIL"}')

## F1: Mini-batch Split Verification

In [ ]:
n_samples = 17; batch_size = 4
idx = np.arange(n_samples)
batches = []
for i in range(0, n_samples, batch_size):
    b = idx[i:i+batch_size]
    batches.append(b)
    print(f'Batch {len(batches)}: {b} (size={len(b)})')

covered = np.array_equal(np.sort(np.concatenate(batches)), np.arange(n_samples))
expected_n = (n_samples + batch_size - 1) // batch_size
print(f'\nBatches: {len(batches)} (expected: {expected_n}) | All covered: {covered}')
print(f'Status: {"PASS" if covered and len(batches) == expected_n else "FAIL"}')

## F2 & F3: Momentum / LR Decay Check

In [ ]:
src = inspect.getsource(Sgd)
has_mom = 'momentum' in src or 'velocity' in src
has_decay = 'decay' in src or 'schedule' in src

print(f'Momentum implemented: {"YES" if has_mom else "NO (not implemented)"}')
print(f'LR decay implemented: {"YES" if has_decay else "NO (not implemented)"}')
print('\nNote: These are optional. Basic SGD (W -= lr * grad) is functional.')